# Paper 2 — Full-Source Hybrid Validation (B channel)

**Objective:** Recalculate `S ∩ B` and `RCI_B` for **NF-ToN-IoT-v2** and **NF-CSE-CIC-IDS2018-v2**, with full-scale structural thresholds **214** and **22**, respectively.

This is a **replication audit** of the original duration-based behavioral channel: its P95/P97 thresholds are estimated using **benign-labeled flows**. It is **not label-free pre-deployment calibration** and is **not independent validation**. No RF/IF training is performed.

**Requires:** `NF-UQ-NIDS-v2.csv` and the two full-scale `entity_table_*.csv` files in your Google Drive folder `paper2_gsd_results`. Each structural file is checked against the corrected counts before doing any behavioral computation. Only aggregate results should be shared for paper drafting; per-source flags stay private.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
# If your full CSV is in Drive rather than /content/nf_uq_nids,
# change DATASET_CSV below to the correct path before execution.

Mounted at /content/drive


In [4]:
"""Paper 2: full-source structural/behavioral intersection audit.

Purpose: Reproduce the previously used B channel (benign-label-calibrated P95/P97
on FLOW_DURATION_MILLISECONDS) against full-source structural candidate sets.

NOT a new independent validation or a label-free calibration experiment.
No raw IPs are printed. Source-level flags are private intermediate artifacts.
"""
from pathlib import Path
import gc
import math
import numpy as np
import pandas as pd

# Set these paths in Colab after mounting Google Drive.
DATASET_CSV = Path('/content/nf_uq_nids/NF-UQ-NIDS-v2.csv')
ENTITY_DIR = Path('/content/drive/MyDrive/paper2_gsd_results')
OUT_DIR = ENTITY_DIR / 'corrected_hybrid_audit'
CHUNK_SIZE = 350_000

# Locked, previously corrected structural policy for these two datasets:
POLICY = {
    'NF-ToN-IoT-v2': {'tau': 214.0, 'expected_n': 23079, 'expected_attack': 31,
                      'expected_selected': 11, 'expected_tp_s': 10},
    'NF-CSE-CIC-IDS2018-v2': {'tau': 22.0, 'expected_n': 224568,
                              'expected_attack': 648, 'expected_selected': 11571,
                              'expected_tp_s': 301},
}
FEATURE = 'FLOW_DURATION_MILLISECONDS'
FIELDS = ['Dataset', 'IPV4_SRC_ADDR', 'Label', FEATURE]
QUANTILES = {'P95': .95, 'P97': .97}


def wilson_interval(success, total, z=1.959963984540054):
    if total == 0:
        return (np.nan, np.nan)
    p = success / total
    denom = 1 + z*z/total
    mid = (p + z*z/(2*total)) / denom
    half = z*math.sqrt(p*(1-p)/total + z*z/(4*total*total))/denom
    return mid-half, mid+half


def metrics(selected, attack, n_sources, n_attack):
    tp = len(selected & attack)
    fp = len(selected) - tp
    fn = n_attack - tp
    precision = tp / (tp+fp) if tp+fp else float('nan')
    recall = tp / n_attack if n_attack else float('nan')
    base_rate = n_attack / n_sources if n_sources else float('nan')
    lift = precision / base_rate if base_rate and np.isfinite(precision) else float('nan')
    lo, hi = wilson_interval(tp, len(selected))
    return dict(flagged=len(selected), TP=tp, FP=fp, FN=fn, precision=precision,
                recall=recall, lift=lift, precision_CI_low=lo, precision_CI_high=hi)


def load_entity_table(dataset_name, policy):
    target = dataset_name.lower().replace('-', '').replace('_', '')
    candidates = list(ENTITY_DIR.rglob('entity_table*.csv'))
    matches = [p for p in candidates if target in p.name.lower().replace('-', '').replace('_', '')]
    if len(matches) != 1:
        raise FileNotFoundError(
            f'{dataset_name}: expected exactly one matching entity_table CSV in {ENTITY_DIR}, '
            f'found {[p.name for p in matches]}. Adjust ENTITY_DIR or remove duplicates.')
    df = pd.read_csv(matches[0], dtype={'src_key':'string', 'out_degree':'float64', 'src_is_attacker':'int8'},
                     usecols=['src_key','out_degree','src_is_attacker'])
    df['src_key'] = df['src_key'].str.strip()
    if df['src_key'].isna().any() or df['src_key'].duplicated().any():
        raise ValueError(f'{dataset_name}: missing or duplicate src_key in entity table')
    n_sources = len(df)
    n_attack = int(df['src_is_attacker'].sum())
    S = set(df.loc[df['out_degree'] >= policy['tau'], 'src_key'].astype(str))
    A = set(df.loc[df['src_is_attacker'] == 1, 'src_key'].astype(str))
    expected = (policy['expected_n'], policy['expected_attack'], policy['expected_selected'], policy['expected_tp_s'])
    got = (n_sources, n_attack, len(S), len(S & A))
    if expected != got:
        raise ValueError(f'{dataset_name}: structural table does not match corrected audit. '
                         f'Expected {expected}; got {got}. Stop rather than mixing cohorts.')
    return df, S, A


def iter_data():
    return pd.read_csv(DATASET_CSV, usecols=FIELDS, chunksize=CHUNK_SIZE,
                       low_memory=False, dtype={'Dataset':'string', 'IPV4_SRC_ADDR':'string'})


def calculate_benign_calibration():
    # This intentionally REPRODUCES the old label-assisted calibration. The label
    # column must never be described as withheld from threshold derivation.
    arrays = {k: [] for k in POLICY}
    observed = {k: 0 for k in POLICY}
    for i, chunk in enumerate(iter_data(), start=1):
        ds = chunk['Dataset'].str.strip()
        attack = pd.to_numeric(chunk['Label'], errors='coerce')
        duration = pd.to_numeric(chunk[FEATURE], errors='coerce')
        for k in POLICY:
            mask = (ds == k) & (attack == 0) & np.isfinite(duration)
            if mask.any():
                arr = duration.loc[mask].to_numpy(dtype=np.float64)
                arrays[k].append(arr)
                observed[k] += arr.size
        if i % 20 == 0:
            print(f'Calibration pass: {i} chunks processed')
    thresholds = {}
    for k in POLICY:
        if not observed[k]:
            raise ValueError(f'No finite benign-labeled {FEATURE} observations found for {k}')
        vals = np.concatenate(arrays[k])
        thresholds[k] = {p: float(np.quantile(vals, q)) for p,q in QUANTILES.items()}
        print(f'{k}: benign-labeled valid flows = {observed[k]:,}; '
              + ', '.join(f'{p} threshold = {t:.6g}' for p,t in thresholds[k].items()))
        del vals
    del arrays
    gc.collect()
    return thresholds, observed


def derive_behavioral_flags(thresholds):
    B = {k: {p:set() for p in QUANTILES} for k in POLICY}
    for i, chunk in enumerate(iter_data(), start=1):
        ds = chunk['Dataset'].str.strip()
        ip = chunk['IPV4_SRC_ADDR'].str.strip()
        val = pd.to_numeric(chunk[FEATURE], errors='coerce')
        for k in POLICY:
            ds_mask = (ds == k) & ip.notna() & (ip != '') & np.isfinite(val)
            if not ds_mask.any():
                continue
            for p, t in thresholds[k].items():
                flagged = ip.loc[ds_mask & (val >= t)].dropna().unique()
                B[k][p].update(f'{k}::{x}' for x in flagged)
        if i % 20 == 0:
            print(f'Inference pass: {i} chunks processed')
    return B


def run():
    if not DATASET_CSV.is_file():
        raise FileNotFoundError(f'{DATASET_CSV} does not exist. Set DATASET_CSV to the full NF-UQ-NIDS-v2 CSV.')
    if not ENTITY_DIR.is_dir():
        raise FileNotFoundError(f'{ENTITY_DIR} does not exist. Set ENTITY_DIR to your paper2_gsd_results folder.')
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {k:load_entity_table(k,v) for k,v in POLICY.items()}
    thresholds, calibration_counts = calculate_benign_calibration()
    B = derive_behavioral_flags(thresholds)
    summary = []
    for name, (entity, S, attack) in tables.items():
        all_keys = set(entity['src_key'].astype(str))
        n = len(all_keys)
        n_attack = len(attack)
        if not S <= all_keys:
            raise AssertionError('Structural set outside matching source cohort')
        for p in QUANTILES:
            unknown = B[name][p] - all_keys
            if unknown:
                raise ValueError(f'{name} {p}: {len(unknown)} B flags not present in the '
                                 'full-scale entity table. Underlying data or cleaning differs; '
                                 'stop and reconcile, do not silently drop missing flags.')
            b = B[name][p]
            hybrid = S & b
            union = S | b
            rci = len(hybrid) / len(union) if union else float('nan')
            for channel, selected in [('S-only', S), ('B-only', b), ('S∩B', hybrid)]:
                summary.append(dict(dataset=name, percentile=p, channel=channel,
                    n_active_sources=n, n_attack_sources=n_attack,
                    source_base_rate=n_attack/n, structural_threshold=POLICY[name]['tau'],
                    benign_calibration_flows=calibration_counts[name],
                    benign_duration_threshold=thresholds[name][p],
                    B_entity_rate=len(b)/n, S_size=len(S), B_size=len(b),
                    intersection_size=len(hybrid), RCI_B=rci, S_subset_B=S<=b,
                    calibration_protocol='BENIGN-LABELED (NOT LABEL-FREE)',
                    **metrics(selected,attack,n,n_attack)))
        # Keep per-source flags in private storage for later RCI / provenance.
        flags = entity[['src_key','out_degree','src_is_attacker']].copy()
        for p in QUANTILES:
            flags[f'B_{p}'] = flags['src_key'].astype(str).isin(B[name][p]).astype(int)
            flags[f'S_and_B_{p}'] = (flags['out_degree'] >= POLICY[name]['tau']) & (flags[f'B_{p}'] == 1)
        flags.to_csv(OUT_DIR/f'{name}_private_source_flags.csv',index=False)
    result = pd.DataFrame(summary)
    result.to_csv(OUT_DIR/'hybrid_fullsource_corrected_summary.csv',index=False)
    print('\n=== SOURCE-LEVEL FULL-COHORT REVALIDATION ===')
    cols=['dataset','percentile','channel','flagged','TP','FP','FN','precision','recall','lift',
          'RCI_B','S_subset_B','B_entity_rate']
    print(result[cols].round(5).to_string(index=False))
    print('\nSaved aggregate summary:',OUT_DIR/'hybrid_fullsource_corrected_summary.csv')
    print('Per-source flags are PRIVATE intermediate files (do not publish or include in manuscript).')
    print('NOTE: This is a replication of the old BENIGN-LABELED B calibration, '
          'not independent/label-free validation.')
    return result



In [5]:
# PRE-FLIGHT: Locate original source data; restore only if necessary.
# This cell must run AFTER mounting Google Drive and AFTER defining run().
from pathlib import Path
import os, shutil, subprocess, sys

DRIVE_ROOT = Path('/content/drive/MyDrive')
FILENAME = 'NF-UQ-NIDS-v2.csv'
search_locations = [
    Path('/content/nf_uq_nids') / FILENAME,
    DRIVE_ROOT / FILENAME,
    DRIVE_ROOT / 'nf_uq_nids' / FILENAME,
]
found = next((p for p in search_locations if p.is_file()), None)
if found is None and DRIVE_ROOT.is_dir():
    # Include possible nested Drive directories.
    found = next(DRIVE_ROOT.rglob(FILENAME), None)

if found is None:
    # The previous Paper 2 notebook downloaded the raw data from Kaggle.
    # Colab's /content disk is temporary across runtime resets.
    key_file = DRIVE_ROOT / 'kaggle.json'
    if not key_file.is_file():
        raise FileNotFoundError(
            'Full NF-UQ-NIDS-v2.csv was not found in the Colab runtime or Drive. '
            'The original Paper 2 notebook used Kaggle credentials. '
            'Place your personal kaggle.json in MyDrive (do not share it), '
            'or manually upload the CSV to MyDrive/nf_uq_nids/ and rerun this cell.'
        )
    print('CSV is absent. Restoring the same Kaggle dataset used by the original Paper 2 notebook.')
    print('This is a LARGE source dataset, not the small stratified sample.')
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    credential_path = kaggle_dir / 'kaggle.json'
    shutil.copyfile(key_file, credential_path)
    credential_path.chmod(0o600)
    if shutil.which('kaggle') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    nf_dir = Path('/content/nf_uq_nids')
    nf_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset',
        '-p', str(nf_dir), '--unzip'
    ], check=True)
    found = next(nf_dir.rglob(FILENAME), None)
    if found is None:
        raise FileNotFoundError(
            'The Kaggle archive was downloaded/extracted, but the expected CSV was not found. '
            f'Inspect directory: {nf_dir} and confirm dataset provenance.'
        )

DATASET_CSV = found
print('Validated dataset path:', DATASET_CSV)
print(f'Raw CSV size: {DATASET_CSV.stat().st_size / 1e9:.2f} GB')
print('Structural entity folder:', ENTITY_DIR)
if not ENTITY_DIR.is_dir():
    raise FileNotFoundError(f'Structural entity tables missing from: {ENTITY_DIR}')

# Check schema on a small portion before the full two-pass read.
sample = pd.read_csv(DATASET_CSV, nrows=3)
missing = sorted(set(FIELDS) - set(sample.columns))
if missing:
    raise ValueError(f'Expected source CSV schema mismatch. Missing: {missing}')
print('Pre-flight passed. Now run the NEXT cell: result = run()')


CSV is absent. Restoring the same Kaggle dataset used by the original Paper 2 notebook.
This is a LARGE source dataset, not the small stratified sample.
Validated dataset path: /content/nf_uq_nids/NF-UQ-NIDS-v2.csv
Raw CSV size: 13.73 GB
Structural entity folder: /content/drive/MyDrive/paper2_gsd_results
Pre-flight passed. Now run the NEXT cell: result = run()


In [6]:
# Change DATASET_CSV and ENTITY_DIR here if necessary.
# Example:
# DATASET_CSV = Path('/content/drive/MyDrive/NF-UQ-NIDS-v2.csv')
result = run()

Calibration pass: 20 chunks processed
Calibration pass: 40 chunks processed
Calibration pass: 60 chunks processed
Calibration pass: 80 chunks processed
Calibration pass: 100 chunks processed
Calibration pass: 120 chunks processed
Calibration pass: 140 chunks processed
Calibration pass: 160 chunks processed
Calibration pass: 180 chunks processed
Calibration pass: 200 chunks processed
NF-ToN-IoT-v2: benign-labeled valid flows = 6,099,469; P95 threshold = 4.29459e+06, P97 threshold = 4.29475e+06
NF-CSE-CIC-IDS2018-v2: benign-labeled valid flows = 16,635,567; P95 threshold = 0, P97 threshold = 4.29495e+06
Inference pass: 20 chunks processed
Inference pass: 40 chunks processed
Inference pass: 60 chunks processed
Inference pass: 80 chunks processed
Inference pass: 100 chunks processed
Inference pass: 120 chunks processed
Inference pass: 140 chunks processed
Inference pass: 160 chunks processed
Inference pass: 180 chunks processed
Inference pass: 200 chunks processed

=== SOURCE-LEVEL FULL-CO

## What to send back

Upload only `hybrid_fullsource_corrected_summary.csv` from `MyDrive/paper2_gsd_results/corrected_hybrid_audit/`. It contains **aggregate** numbers, no IP addresses. Please keep the `*_private_source_flags.csv` files in Drive unless specifically needed for local verification.

If a cohort mismatch exception appears, send the **error text** instead of weakening the assertion; it indicates the flow file and structural table are not aligned.